In [34]:
# ================================================
# 0 — IMPORTS
# ================================================
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import GroupKFold
from catboost import CatBoostClassifier, Pool
import optuna

# ================================================
# 1 — LOAD
# ================================================
train = pd.read_csv("datasets/training_data.csv", encoding="latin1")
test = pd.read_csv("datasets/test_data.csv", encoding="latin1")

y = train["AVERAGE_SPEED_DIFF"].fillna("None").astype(str)
X = train.drop("AVERAGE_SPEED_DIFF", axis=1)

# ================================================
# 2 — SAFE DATE FEATURES (ideal p/ dataset pequeno)
# ================================================
def process_dates(df):
    df = df.copy()
    df["record_date"] = pd.to_datetime(df["record_date"])

    df["month"] = df["record_date"].dt.month
    df["hour"] = df["record_date"].dt.hour
    df["weekday"] = df["record_date"].dt.weekday

    df = df.drop(columns=["record_date"])
    return df

X = process_dates(X)
test = process_dates(test)

# ================================================
# 3 — FIX NUMERIC COLUMNS
# ================================================
numeric_cols = [
    "AVERAGE_FREE_FLOW_SPEED",
    "AVERAGE_TIME_DIFF",
    "AVERAGE_FREE_FLOW_TIME",
    "AVERAGE_TEMPERATURE",
    "AVERAGE_ATMOSP_PRESSURE",
    "AVERAGE_HUMIDITY",
    "AVERAGE_WIND_SPEED",
    "AVERAGE_PRECIPITATION"
]

for c in numeric_cols:
    X[c] = pd.to_numeric(X[c], errors="coerce").fillna(0)
    test[c] = pd.to_numeric(test[c], errors="coerce").fillna(0)

# ================================================
# 4 — LIGHT FEATURE ENGINEERING (BEST FOR SMALL DATASET)
# ================================================
def add_features(df):
    df = df.copy()

    df["is_weekend"] = (df["weekday"] >= 5).astype(int)

    df["rush_hour"] = (
        df["hour"].between(7, 9) |
        df["hour"].between(17, 19)
    ).astype(int)

    df["time_ratio"] = (df["AVERAGE_TIME_DIFF"] + 1e-6) / (df["AVERAGE_FREE_FLOW_TIME"] + 1e-6)
    df["weather_index"] = df["AVERAGE_PRECIPITATION"] * df["AVERAGE_HUMIDITY"]

    return df

X = add_features(X)
test = add_features(test)

# ================================================
# 5 — CLEAN CATEGORICALS
# ================================================
def clean_cats(df):
    df = df.copy()
    for col in df.columns:
        if df[col].dtype == object:
            df[col] = df[col].astype(str).str.strip()
            df[col] = df[col].replace("", "Unknown").fillna("Unknown")
    return df

X = clean_cats(X)
test = clean_cats(test)

cat_cols = [c for c in X.columns if X[c].dtype == object]
cat_idx = [X.columns.get_loc(c) for c in cat_cols]

# ================================================
# 6 — REAL VALIDATION USING GROUP KFOLD 
#    (NO DATA LEAKAGE)
# ================================================
groups = X["month"]  # melhor separação temporal

kf = GroupKFold(n_splits=5)

# =======================
# INITIAL TRUE VALIDATION
# =======================
def evaluate_params(params):
    scores = []

    for train_idx, val_idx in kf.split(X, y, groups):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

        train_pool = Pool(X_tr, y_tr, cat_features=cat_idx)
        val_pool = Pool(X_val, y_val, cat_features=cat_idx)

        model = CatBoostClassifier(**params)
        model.fit(train_pool, eval_set=val_pool, verbose=False)

        pred = model.predict(X_val).reshape(-1)
        scores.append(accuracy_score(y_val, pred))

    return np.mean(scores)

# ==========================================================
# 7 — OPTUNA — tuned for SMALL dataset (fast + accurate)
# ==========================================================
def objective(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 400, 900),
        "depth": trial.suggest_int("depth", 6, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.09),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 5, 30),
        "bootstrap_type": "Bayesian",
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.5, 5),

        "loss_function": "MultiClass",
        "eval_metric": "Accuracy",
        "random_seed": 42,
        "od_type": "Iter",
        "od_wait": 40,
        "verbose": False
    }

    return evaluate_params(params)

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=35, show_progress_bar=True)

best_params = study.best_params
print("\nBEST PARAMS FOUND:", best_params)

# ==========================================================
# 8 — FINAL VALIDATION (TRUE SCORE!)
# ==========================================================
true_acc = evaluate_params(best_params)

print("\n==============================")
print(f"TRUE VALIDATION ACC (NO LEAK): {true_acc:.6f}")
print("==============================\n")

# ==========================================================
# 9 — TRAIN FINAL MODEL ON 100% DATASET
# ==========================================================
final_model = CatBoostClassifier(
    **best_params,
    loss_function="MultiClass",
    eval_metric="Accuracy",
    random_seed=42
)

full_pool = Pool(X, y, cat_features=cat_idx)
final_model.fit(full_pool, verbose=100)

# ==========================================================
# 10 — SUBMISSION (ID 1..1500 FIXED)
# ==========================================================
test_pool = Pool(test, cat_features=cat_idx)
test_pred = final_model.predict(test_pool).reshape(-1)

submission = pd.DataFrame({
    "RowId": np.arange(1, len(test_pred) + 1),
    "Speed_Diff": test_pred
})

submission.to_csv("submission.csv", index=False)
print("✔ submission.csv criado com sucesso!")


[I 2025-11-28 10:54:40,432] A new study created in memory with name: no-name-93832ebb-cdd2-4768-a830-f45c098fc161
Best trial: 0. Best value: 0.783962:   3%|▎         | 1/35 [00:15<08:46, 15.50s/it]

[I 2025-11-28 10:54:55,934] Trial 0 finished with value: 0.783962188858701 and parameters: {'iterations': 423, 'depth': 7, 'learning_rate': 0.053197653208179055, 'l2_leaf_reg': 16.394120468540834, 'bagging_temperature': 2.880170757137325}. Best is trial 0 with value: 0.783962188858701.


Best trial: 1. Best value: 0.789259:   6%|▌         | 2/35 [00:34<09:40, 17.58s/it]

[I 2025-11-28 10:55:14,972] Trial 1 finished with value: 0.7892588152225246 and parameters: {'iterations': 634, 'depth': 6, 'learning_rate': 0.06192860295563271, 'l2_leaf_reg': 24.808792194010543, 'bagging_temperature': 0.7109853652250465}. Best is trial 1 with value: 0.7892588152225246.


Best trial: 1. Best value: 0.789259:   9%|▊         | 3/35 [00:56<10:23, 19.49s/it]

...

[I 2025-11-28 11:13:32,763] Trial 32 finished with value: 0.7879012689199962 and parameters: {'iterations': 623, 'depth': 7, 'learning_rate': 0.07971446362149386, 'l2_leaf_reg': 24.88888250213287, 'bagging_temperature': 0.737978662226414}. Best is trial 10 with value: 0.7915470912581206.


Best trial: 10. Best value: 0.791547:  97%|█████████▋| 34/35 [19:18<00:18, 18.78s/it]

[I 2025-11-28 11:13:59,318] Trial 33 finished with value: 0.7899407444562856 and parameters: {'iterations': 447, 'depth': 6, 'learning_rate': 0.08486872785854228, 'l2_leaf_reg': 26.896713779795018, 'bagging_temperature': 0.5091011264584638}. Best is trial 10 with value: 0.7915470912581206.


Best trial: 10. Best value: 0.791547: 100%|██████████| 35/35 [19:40<00:00, 33.72s/it]


[I 2025-11-28 11:14:20,628] Trial 34 finished with value: 0.7889195896291457 and parameters: {'iterations': 567, 'depth': 7, 'learning_rate': 0.08950882910688138, 'l2_leaf_reg': 21.632912724662717, 'bagging_temperature': 1.478753199529185}. Best is trial 10 with value: 0.7915470912581206.

BEST PARAMS FOUND: {'iterations': 602, 'depth': 8, 'learning_rate': 0.0872227617973595, 'l2_leaf_reg': 26.784736964013252, 'bagging_temperature': 0.5631238783771639}

TRUE VALIDATION ACC (NO LEAK): 0.789549

0:	learn: 0.7475044	total: 35.1ms	remaining: 21.1s
100:	learn: 0.8134175	total: 3.23s	remaining: 16s
200:	learn: 0.8391075	total: 6.16s	remaining: 12.3s
300:	learn: 0.8521726	total: 9.16s	remaining: 9.16s
400:	learn: 0.8617146	total: 12.2s	remaining: 6.12s
500:	learn: 0.8706694	total: 15.3s	remaining: 3.09s
600:	learn: 0.8783030	total: 18.3s	remaining: 30.4ms
601:	learn: 0.8785966	total: 18.3s	remaining: 0us
✔ submission.csv criado com sucesso!
